In [2]:
import torch
from occhio.autoencoder import TiedLinearRelu
from occhio.distributions.hierarchical import HierarchicalSparse
from occhio.model_grid import ModelGrid, Axis
from occhio.toy_model import ToyModel
from occhio.visualization import *

device = "mps"

In [3]:
N_FEATURES, N_HIDDEN = 5, 2

axis_embed = Axis(label="p_base", values=torch.logspace(-1, 0, 6))

def create_embedding_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=3,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.9 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_embed = ModelGrid(create_embedding_model, axes=[axis_embed])

Initializing models:   0%|          | 0/6 [00:00<?, ?model/s]

Grouping distributions: 100%|██████████| 6/6 [00:00<00:00, 1610.20model/s]


In [4]:
grid_embed.fit(batch_size=2048, n_epochs=20_000)

Grouping distributions: 100%|██████████| 6/6 [00:00<00:00, 2627.19model/s]


In [5]:
print(axis_embed.values)
plot_embedding(grid_embed)


tensor([0.1000, 0.1585, 0.2512, 0.3981, 0.6310, 1.0000])


In [6]:
N_FEATURES, N_HIDDEN = 5, 2

axis_importance = Axis(label="Importance", values=torch.logspace(-1, 1, 70))
axis_density    = Axis(label="p_base",     values=torch.logspace(-1, 0,  70))

def create_phase_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=3,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=float(params["Importance"]) ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_phase = ModelGrid(create_phase_model, axes=[axis_importance, axis_density], cache_samples=True)

Grouping distributions: 100%|██████████| 4900/4900 [00:00<00:00, 7938.94model/s]


In [7]:
grid_phase.fit(batch_size=216, n_epochs=10_000)

Grouping distributions: 100%|██████████| 4900/4900 [00:00<00:00, 7680.96model/s]


In [18]:
import pickle

# Save the fitted grid_phase to disk for reuse later
with open("grid_phase.pkl", "wb") as f:
    pickle.dump(grid_phase, f)

# You can load and reconstruct the object like this:
with open("grid_phase.pkl", "rb") as f:
    grid_phase_loaded = pickle.load(f)
    # Now grid_phase_loaded is a reconstructed ModelGrid object


In [8]:
# This feature is always active
plot_phase_change(grid_phase, tracked_feature=0)  

In [9]:
plot_phase_change(grid_phase, tracked_feature=1)  # depth-1 child

In [10]:
plot_phase_change(grid_phase, tracked_feature=2)

In [ ]:
plot_phase_change(grid_phase, tracked_feature=3)

In [12]:
plot_phase_change(grid_phase, tracked_feature=4)  # depth-2 leaf — sparsest

In [15]:
# Exp 3 — Geometry: 100 features, 20 hidden (5:1), sweep p_base
# Flat importance so geometry is driven by density alone
N_FEATURES, N_HIDDEN = 100, 20

axis_geo = Axis(label="p_base", values=torch.logspace(-2, 0, 16))

def create_geometry_model(params):
    return ToyModel(
        distribution=HierarchicalSparse(
            N_FEATURES, p_base=float(params["p_base"]), depth_decay=0.85, max_children=4,
            device=device, generator=torch.Generator(device=device).manual_seed(42),
        ),
        importances=0.999 ** torch.arange(N_FEATURES),
        ae=TiedLinearRelu(N_FEATURES, N_HIDDEN, device=device,
                          generator=torch.Generator(device=device).manual_seed(7)),
    )

grid_geo = ModelGrid(create_geometry_model, axes=[axis_geo])

Grouping distributions: 100%|██████████| 16/16 [00:00<00:00, 2604.15model/s]


In [16]:
grid_geo.fit(batch_size=2048, n_epochs=10_000)

Grouping distributions: 100%|██████████| 16/16 [00:00<00:00, 4144.06model/s]


In [17]:
plot_geometry(grid_phase)

AssertionError: plot_geometry only supports ModelGrids with exactly one non-singleton dimension.Got shape (70, 70) with 2 non-singleton dims at indices [0 1]).

In [ ]:
plot_geometry(grid_geo)

NameError: name 'grid_geo' is not defined

In [ ]:
m